# 01 - Random Forest (pixel classifier)

Per-pixel baseline. Features: the 6 Sentinel-2 bands plus NDVI, NBR and NDWI.

- Trains on a random **sample of labeled pixels** from the train blocks (2019, 2021, 2023 pooled).
- Chooses the decision threshold on the **val** blocks, then reports precision / recall / F1 / IoU per year on the **test** blocks.
- Uses the split from `00_data_prep.ipynb`, so the numbers can be compared with the U-Net.

Label meaning (to confirm with the data owner): `1` = not yet cleared in that year, `0` = cleared. Only labeled pixels are used.

In [1]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, glob
src_dir = "/content/drive/MyDrive/PreyLang"   # the folder with copy of the tiles
os.makedirs("/content/tiles", exist_ok=True)
for f in sorted(glob.glob(f"{src_dir}/*.tif")):
    dst = f"/content/tiles/{os.path.basename(f)}"
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f)):
        shutil.copy(f, dst)
    print("OK " if os.path.getsize(dst) == os.path.getsize(f) else "BAD", os.path.basename(f))

Mounted at /content/drive
OK  PreyLang_Composite_2019_tile_0_0.tif
OK  PreyLang_Composite_2019_tile_0_1.tif
OK  PreyLang_Composite_2019_tile_1_0.tif
OK  PreyLang_Composite_2019_tile_1_1.tif
OK  PreyLang_Composite_2021_tile_0_0.tif
OK  PreyLang_Composite_2021_tile_0_1.tif
OK  PreyLang_Composite_2021_tile_1_0.tif
OK  PreyLang_Composite_2021_tile_1_1.tif
OK  PreyLang_Composite_2023_tile_0_0.tif
OK  PreyLang_Composite_2023_tile_0_1.tif
OK  PreyLang_Composite_2023_tile_1_0.tif
OK  PreyLang_Composite_2023_tile_1_1.tif
OK  PreyLang_Composite_2025_tile_0_0.tif
OK  PreyLang_Composite_2025_tile_0_1.tif
OK  PreyLang_Composite_2025_tile_1_0.tif
OK  PreyLang_Composite_2025_tile_1_1.tif


In [2]:
!pip -q install rasterio

In [3]:
import os, re, json, glob, time
import numpy as np, pandas as pd
import rasterio
from rasterio.windows import Window

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

# ---------------- CONFIG (keep identical in all three notebooks) ----------------
TILE_DIR = "/content/tiles"                         # local copy of the 16 TIFFs (not the Drive mount)
WORK_DIR = "/content/drive/MyDrive/PreyLang_work"   # splits, stats, models and metrics are saved here
TILES    = ["0_0", "0_1", "1_0", "1_1"]
TRAIN_YEARS = [2019, 2021, 2023]                    # 2025 has no positive labels, so it is not used
BLOCK    = 1024      # spatial block size in pixels (about 10 km). A block never crosses a split.
PATCH    = 256       # U-Net patch size
SEED     = 42
BANDS    = ["B2", "B3", "B4", "B8", "B11", "B12"]
os.makedirs(WORK_DIR, exist_ok=True)
# --------------------------------------------------------------------------------

def tile_path(year, tile):
    return f"{TILE_DIR}/PreyLang_Composite_{year}_tile_{tile}.tif"

def read_window(year, tile, r0, c0, h, w):
    """Float32 array (7, h, w): B2 B3 B4 B8 B11 B12 label (NaN = missing)."""
    with rasterio.open(tile_path(year, tile)) as src:
        return src.read(window=Window(c0, r0, w, h)).astype("float32")

def add_indices(b):
    """b: (6, ...) ordered B2 B3 B4 B8 B11 B12 -> (3, ...) NDVI, NBR, NDWI."""
    eps = 1e-6
    ndvi = (b[3] - b[2]) / (b[3] + b[2] + eps)
    nbr  = (b[3] - b[5]) / (b[3] + b[5] + eps)
    ndwi = (b[1] - b[3]) / (b[1] + b[3] + eps)
    return np.stack([ndvi, nbr, ndwi])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.inspection import permutation_importance
import joblib

splits = json.load(open(f"{WORK_DIR}/splits.json"))
assert splits["block"] == BLOCK and splits["seed"] == SEED, "config differs from 00_data_prep"
by_split = {s: [b for b in splits["blocks"] if b["split"] == s] for s in ["train", "val", "test"]}
print({s: len(v) for s, v in by_split.items()}, "blocks")

FEATURES = BANDS + ["NDVI", "NBR", "NDWI"]

def make_features(bands):                 # (6, N) -> (N, 9)
    return np.concatenate([bands, add_indices(bands)], 0).T.astype("float32")

def sample_pixels(block_list, frac, rng):
    """Random fraction of the valid labeled pixels of every block and year."""
    X, Y, YR = [], [], []
    for b in block_list:
        for yr in TRAIN_YEARS:
            a = read_window(yr, b["tile"], b["r0"], b["c0"], b["h"], b["w"])
            bands, lab = a[:6].reshape(6, -1), a[6].ravel()
            idx = np.flatnonzero(np.isfinite(bands).all(0) & np.isfinite(lab))
            if len(idx) == 0:
                continue
            sel = rng.choice(idx, max(1, int(len(idx) * frac)), replace=False)
            X.append(make_features(bands[:, sel]))
            Y.append(lab[sel].astype("int8"))
            YR.append(np.full(len(sel), yr, dtype="int16"))
    return np.concatenate(X), np.concatenate(Y), np.concatenate(YR)

{'train': 38, 'val': 7, 'test': 4} blocks


## 1. Sample pixels
`TRAIN_FRAC` / `EVAL_FRAC` are the share of valid labeled pixels kept. A uniform random sample gives an unbiased estimate of the full-image metrics.

In [5]:
TRAIN_FRAC, EVAL_FRAC = 0.02, 0.05
rng = np.random.default_rng(SEED)
t0 = time.time()
Xtr, ytr, yrtr = sample_pixels(by_split["train"], TRAIN_FRAC, rng)
Xva, yva, yrva = sample_pixels(by_split["val"],   EVAL_FRAC,  rng)
Xte, yte, yrte = sample_pixels(by_split["test"],  EVAL_FRAC,  rng)
print(f"{time.time()-t0:.0f}s  train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}")
for name, y, yr in [("train", ytr, yrtr), ("val", yva, yrva), ("test", yte, yrte)]:
    print(name, {int(k): round(float(y[yr == k].mean()), 3) for k in TRAIN_YEARS}, "(positive rate per year)")

47s  train (514143, 9)  val (250905, 9)  test (250509, 9)
train {2019: 0.287, 2021: 0.17, 2023: 0.06} (positive rate per year)
val {2019: 0.236, 2021: 0.151, 2023: 0.042} (positive rate per year)
test {2019: 0.167, 2021: 0.108, 2023: 0.044} (positive rate per year)


## 2. Train

In [6]:
rf = RandomForestClassifier(n_estimators=150, max_depth=20, min_samples_leaf=5, max_features="sqrt",
                            class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
t0 = time.time()
rf.fit(Xtr, ytr)
print(f"trained in {time.time()-t0:.0f}s")

trained in 350s


## 3. Pick the threshold on val, evaluate on test

In [7]:
def prf(y, pred):
    y, pred = y.astype(bool), pred.astype(bool)
    tp = int((pred & y).sum()); fp = int((pred & ~y).sum()); fn = int((~pred & y).sum()); tn = int((~pred & ~y).sum())
    return dict(precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                f1=2 * tp / max(2 * tp + fp + fn, 1), iou=tp / max(tp + fp + fn, 1),
                accuracy=(tp + tn) / max(len(y), 1), n=int(len(y)), positives=int(y.sum()))

def report(y, p, yr, thr):
    rows = {}
    for k in TRAIN_YEARS + ["all"]:
        m = np.ones(len(y), bool) if k == "all" else (yr == k)
        r = prf(y[m], p[m] >= thr)
        r["pr_auc"] = average_precision_score(y[m], p[m]) if y[m].any() else float("nan")
        r["prevalence"] = float(y[m].mean())
        rows[str(k)] = r
    return pd.DataFrame(rows).T.round(3)

pva = rf.predict_proba(Xva)[:, 1]
pte = rf.predict_proba(Xte)[:, 1]
grid = np.linspace(0.05, 0.95, 19)
f1s = [prf(yva, pva >= t)["f1"] for t in grid]
thr = float(grid[int(np.argmax(f1s))])
print(f"best threshold on val: {thr:.2f}  (val F1 {max(f1s):.3f})")

print("\nTEST @ 0.50")
res_050 = report(yte, pte, yrte, 0.5); print(res_050)
print(f"\nTEST @ {thr:.2f} (tuned on val)")
res_tuned = report(yte, pte, yrte, thr); print(res_tuned)
print("\nA model that predicts 'all positive' would get F1 = 2p/(1+p) where p is the prevalence column.")

best threshold on val: 0.70  (val F1 0.707)

TEST @ 0.50
      precision  recall     f1    iou  accuracy         n  positives  pr_auc  \
2019      0.595   0.695  0.641  0.472     0.870   83503.0    13958.0   0.698   
2021      0.528   0.667  0.589  0.418     0.899   83503.0     9029.0   0.624   
2023      0.273   0.446  0.339  0.204     0.924   83503.0     3658.0   0.254   
all       0.515   0.651  0.575  0.404     0.898  250509.0    26645.0   0.604   

      prevalence  
2019       0.167  
2021       0.108  
2023       0.044  
all        0.106  

TEST @ 0.70 (tuned on val)
      precision  recall     f1    iou  accuracy         n  positives  pr_auc  \
2019      0.705   0.591  0.643  0.473     0.890   83503.0    13958.0   0.698   
2021      0.650   0.573  0.609  0.438     0.920   83503.0     9029.0   0.624   
2023      0.355   0.335  0.345  0.208     0.944   83503.0     3658.0   0.254   
all       0.633   0.550  0.589  0.417     0.918  250509.0    26645.0   0.604   

      prevalence  

## 4. Feature importance (permutation, on val) and save

In [8]:
s = np.random.default_rng(SEED).choice(len(yva), min(30000, len(yva)), replace=False)
pi = permutation_importance(rf, Xva[s], yva[s], scoring="average_precision", n_repeats=3,
                            random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print(imp.round(4))

joblib.dump(rf, f"{WORK_DIR}/rf_model.joblib", compress=3)
json.dump({"threshold": thr, "test_at_0.5": res_050.to_dict("index"), "test_tuned": res_tuned.to_dict("index"),
           "permutation_importance": imp.to_dict(), "features": FEATURES,
           "train_frac": TRAIN_FRAC, "eval_frac": EVAL_FRAC},
          open(f"{WORK_DIR}/rf_metrics.json", "w"), indent=1)
print("saved rf_model.joblib and rf_metrics.json")

B11     0.4291
B12     0.1021
B4      0.0553
NBR     0.0550
B3      0.0488
NDVI    0.0446
B2      0.0406
NDWI    0.0292
B8      0.0242
dtype: float64
saved rf_model.joblib and rf_metrics.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')